# Carga de datos

Materializa la tabla de trabajo y verifica que llego como se espera.

En produccion este notebook no existiria: la informacion vendria ya del Data Warehouse. Ocupa ese
lugar porque la fuente es un `.csv` de ejemplo.

| | `Cargar_datos.ipynb` | `comprension_eda.ipynb` |
|---|---|---|
| Pregunta | ¿Llegaron bien los datos? | ¿Que dicen los datos? |
| Hace | Ingesta y validacion | Limpieza y exploracion |
| Produce | Diagnostico de calidad | `Base_de_datos_limpia.csv` |

Es la compuerta de calidad: mide el estado del dato crudo y con eso justifica la limpieza posterior.

In [1]:
# Importación de librerías
import json
import sys
from pathlib import Path

import pandas as pd

# --- Localizacion de la raiz del proyecto ---
# El notebook vive en mlops_pipeline/ y los CSV en la raiz del repo. En lugar de
# fijar '../' a mano (que se rompe si el notebook se mueve o si el kernel
# arranca en otro directorio), se busca hacia arriba el archivo de datos crudos.
def encontrar_raiz(marcador='Base_de_datos.csv', max_niveles=6):
    """Sube por el arbol de directorios hasta encontrar el archivo marcador."""
    actual = Path.cwd().resolve()
    for _ in range(max_niveles + 1):
        if (actual / marcador).exists():
            return actual
        if actual.parent == actual:      # se llego a la raiz del sistema
            break
        actual = actual.parent
    raise FileNotFoundError(
        f"No se encontro '{marcador}' subiendo desde {Path.cwd().resolve()}. "
        "Verifica que el notebook este dentro del repositorio del proyecto.")


RUTA_RAIZ = encontrar_raiz()
RUTA_CRUDO = RUTA_RAIZ / 'Base_de_datos.csv'
RUTA_LIMPIO = RUTA_RAIZ / 'Base_de_datos_limpia.csv'

# El contrato de negocio se importa del modulo hermano, no se redefine aqui.
sys.path.insert(0, str(RUTA_RAIZ / 'mlops_pipeline'))
import reglas_negocio as rn

print("Raiz del proyecto:", RUTA_RAIZ)
print("Reglas del contrato:", len(rn.REGLAS_VALIDACION))

Raiz del proyecto: C:\Users\JOARM\MLOPS_CURSE
Reglas del contrato: 22


## La fuente

Dos detalles del archivo hay que declararlos, porque los valores por defecto de pandas fallan con
ambos.

**Separador `;`** en lugar de coma. Es la convencion de exportacion en configuraciones donde la coma
es el separador decimal.

**Codificacion `utf-8-sig`**. El archivo trae BOM. Sin declararlo, la primera columna llega como
`﻿tipo_credito` y falla cualquier acceso por nombre.

Los dos estan en `config.json`, no escritos a mano en cada notebook.

In [2]:
CONFIG = json.loads((RUTA_RAIZ / 'config.json').read_text(encoding='utf-8'))
SEP = CONFIG['data']['separator']
ENC = CONFIG['data']['encoding']

print(f"Separador   : {SEP!r}")
print(f"Codificacion: {ENC!r}")
print(f"Objetivo    : {CONFIG['target_variable']}")

df = pd.read_csv(RUTA_CRUDO, sep=SEP, encoding=ENC)
print(f"\nDimensiones : {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

Separador   : ';'
Codificacion: 'utf-8-sig'
Objetivo    : Pago_atiempo

Dimensiones : 10,763 filas x 23 columnas


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,7/01/2025 14:40,1852560,12,32,Empleado,3500000,1000000,128650,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,9/01/2025 11:18,3181080,6,34,Empleado,5000000,2000000,441817,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,26/12/2024 18:52,670200,5,34,Independiente,5000000,2000000,108632,"95,227787",...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,4/12/2024 14:20,506807,2,25,Independiente,3000000,900000,199684,"95,227787",...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,30/04/2025 18:41,999780,10,26,Empleado,2000000,600000,92509,"95,227787",...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


## Verificacion estructural

Antes de mirar el contenido se comprueba la forma de la tabla: tipos, nulos y peso en memoria.

Hay un dato que condiciona el proyecto entero y conviene registrarlo desde la ingesta: **no existe
identificador de cliente**. La unidad de observacion es el credito, no la persona. Eso impide una
particion agrupada por cliente y debe declararse al reportar cualquier metrica.

In [3]:
resumen = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'nulos_%': (df.isna().mean() * 100).round(2),
    'unicos': df.nunique(),
})
resumen.index.name = 'columna'
print(resumen.to_string())

print(f"\nMemoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")
print(f"Identificador de cliente: NO existe en el dataset")

                                  tipo  nulos  nulos_%  unicos
columna                                                       
tipo_credito                     int64      0     0.00       6
fecha_prestamo                     str      0     0.00   10444
capital_prestado                 int64      0     0.00    7306
plazo_meses                      int64      0     0.00      18
edad_cliente                     int64      0     0.00      54
tipo_laboral                       str      0     0.00       2
salario_cliente                  int64      0     0.00    1385
total_otros_prestamos            int64      0     0.00    1538
cuota_pactada                    int64      0     0.00    9836
puntaje                            str      0     0.00     248
puntaje_datacredito            float64      6     0.06     315
cant_creditosvigentes            int64      0     0.00      39
huella_consulta                  int64      0     0.00      28
saldo_mora                     float64    156     1.45 

## El contrato como compuerta de calidad

`reglas_negocio.py` publica `REGLAS_VALIDACION`: rango valido, dominio permitido y fuente documental
de cada variable. Se aplica aqui sobre el dato crudo para medir en que estado llega.

Sobre el orden: estas reglas se derivaron en `comprension_eda.ipynb`, que corre despues. No es
circular. El contrato es un artefacto de la Fase 1 que todas las etapas aplican, igual que hara
`model_deploy.py` con cada cliente nuevo. Aqui se aplica, no se deriva.

Se validan las dos versiones del dataset para medir el efecto de la limpieza.

In [4]:
violaciones_crudo = rn.validar_dataframe(df)

print(f"=== DATO CRUDO: {len(violaciones_crudo)} violaciones ===")
for v in violaciones_crudo:
    print("   -", v)

if RUTA_LIMPIO.exists():
    df_limpio = pd.read_csv(RUTA_LIMPIO, sep=SEP, encoding=ENC)
    violaciones_limpio = rn.validar_dataframe(df_limpio)
    print(f"\n=== TRAS LA LIMPIEZA: {len(violaciones_limpio)} violaciones ===")
    for v in violaciones_limpio:
        print("   -", v)
else:
    print("\n(Base_de_datos_limpia.csv aun no existe: ejecuta comprension_eda.ipynb)")

=== DATO CRUDO: 12 violaciones ===
   - edad_cliente: 150 valores sobre el maximo (90)
   - salario_cliente: 24 valores bajo el minimo (1)
   - salario_cliente: 193 valores sobre el maximo (20000000)
   - puntaje_datacredito: 147 valores bajo el minimo (150)
   - puntaje_datacredito: 6 valores sobre el maximo (950)
   - saldo_mora: 156 nulos no permitidos
   - saldo_total: 156 nulos no permitidos
   - saldo_principal: 405 nulos no permitidos
   - saldo_mora_codeudor: 590 nulos no permitidos
   - tendencia_ingresos: 2932 nulos no permitidos
   - tendencia_ingresos: 58 valores fuera del dominio permitido
   - total_otros_prestamos: 13 valores sobre el maximo (1000000000)

=== TRAS LA LIMPIEZA: 1 violaciones ===
   - total_otros_prestamos: 13 valores sobre el maximo (1000000000)


### Lectura del resultado

El dato crudo incumple el contrato en **12 puntos**. Cada uno tiene su tratamiento documentado en
`comprension_eda.ipynb`.

| Violacion | Que es | Tratamiento |
|---|---|---|
| `puntaje_datacredito`: 147 bajo minimo, 6 sobre maximo | 153 registros fuera del rango oficial `[150, 950]`. Los ceros no son puntajes, son ausencia de historial | Se separan como `SIN_DATO`, no se imputan |
| `edad_cliente`: 150 sobre 90 anios | Errores de captura, con brecha visible en la distribucion entre 70 y 120 | Correccion con bandera de trazabilidad |
| `salario_cliente`: 24 bajo minimo, 193 sobre maximo | Valores imposibles y atipicos extremos | Correccion con bandera |
| `tendencia_ingresos`: 2.932 nulos, 58 corruptos | Dominio cerrado con valores fuera de el | Reconstruccion con bandera |
| Nulos en los cuatro saldos | Ausencia de reporte del buro | La ausencia es informativa: se conserva como categoria |
| `total_otros_prestamos`: 13 sobre 1.000M | Montos no verificables | Se señalan, no se imputan |

Tras la limpieza queda **una sola violacion**, y es deliberada: los 13 registros no verificables de
`total_otros_prestamos`. La regla los marca en cada ejecucion en vez de dejarlos pasar en silencio.
Que sobreviva es la prueba de que la compuerta sigue activa.

Un detalle que solo aparece validando el dato crudo: el origen trae las fechas en formato `d/m/Y`
(`7/01/2025` es el 7 de enero) y el dataset procesado las trae en ISO. Sin declarar la convencion,
pandas lee al reves las fechas ambiguas, desplaza el rango del dataset y reporta 21 fechas futuras
inexistentes. El contrato lo resuelve con dos pasadas deterministas: ISO primero, formato de origen
para lo que no encaje.

## Cierre

La tabla esta materializada y su calidad de origen medida. Sigue `comprension_eda.ipynb`, que
explora, limpia y produce `Base_de_datos_limpia.csv`, insumo de `ft_engineering.py`.

`Cargar_datos.ipynb` → `comprension_eda.ipynb` → `ft_engineering.py` → `hueristic_model.py`